# CS 483/583 · Module 12 · Language Modeling and Large Language Models

The desk log contains short entries such as “open desk” and “close door.” After the first word, how should a model divide its probability among possible continuations? Follow one prediction from a pair of words to a loss and a weight update.

Module 11 produced a vector at each position using only the accessible prefix. Now give that vector a job. A final linear layer produces one score per vocabulary item. We begin with a one-token context so you can inspect every score by hand.

## Running the examples

Use a Python 3 notebook with PyTorch installed. A CPU is sufficient. Run the cells in order, then change an input and predict what will happen.

If a model reads the word at position three while training to predict position three, what goes wrong?

<details><summary>Reveal answer</summary>

The input would already contain the answer. Use earlier positions to predict position three, or shift the targets one step ahead of each input.

</details>

## Choose the units to predict

A tokenizer splits a text into units and assigns integer identifiers. A vocabulary lists the units the model knows. Our miniature tokenizer splits on spaces and treats each word as a token. Other tokenizers can break a word into several pieces. The tokenizer must be shared by training and evaluation.

For the vocabulary open, close, desk, door and <end>, “open desk <end>” becomes [0, 2, 4]. A token ID is a lookup address. The numerical difference between two IDs has no linguistic meaning.

**Formal notation.** V is the vocabulary size. An observed sequence is t₁, t₂, …, tₙ, where tᵢ is the token identifier in position i.

**Predict before you run.** Could we swap the IDs of open and door without changing what the text says?

In [1]:
vocab = ["open", "close", "desk", "door", "<end>"]
ids = {word: i for i, word in enumerate(vocab)}
text = "open desk <end>"
tokens = [ids[word] for word in text.split()]
print(tokens)
print(" ".join(vocab[i] for i in tokens))

[0, 2, 4]
open desk <end>


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

Yes, if we update the encoding, decoding and any model lookup rows consistently. The assignment of numbers is a convention.

</details>

## Turn text into training pairs

A next-token model reads a prefix and predicts what follows. Multiple examples come from a single sequence by shifting the targets one position to the left. A bigram model keeps only the last token of each prefix. A masked transformer can use the whole available prefix up to its context limit.

For open, desk, <end>, the input open has target desk. The prefix open desk has target <end>. A bigram model sees desk for this second prediction. We condition on the provided first word in this example.

**Formal notation.** p(tᵢ₊₁ | t₁,…,tᵢ) means the probability assigned to the next token given the prefix. A bigram approximation uses p(tᵢ₊₁ | tᵢ).

**Predict before you run.** How many next-token predictions are possible in a five-token sequence if its first token is given?

In [2]:
words = ["open", "desk", "<end>"]
for i in range(len(words) - 1):
    prefix, target = words[:i+1], words[i+1]
    print("prefix", prefix, "bigram input", prefix[-1], "target", target)

prefix ['open'] bigram input open target desk
prefix ['open', 'desk'] bigram input desk target <end>


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

Four. Positions one through four each have a next token in the sequence.

</details>

## Read the distribution over words

A model first produces unnormalized real scores called logits. Softmax turns them into probabilities. The observed next word picks which probability counts toward the loss. For easy arithmetic, consider a toy prediction step with exactly three possible outputs.

Set the logits for desk, door and shelf to ln(6), ln(3) and ln(1). Exponentiation gives weights 6, 3, 1. Dividing by ten gives 0.6, 0.3, 0.1. If door follows, the loss is −ln(0.3), about 1.204.

**Formal notation.** pⱼ = exp(zⱼ)/Σₖ exp(zₖ), where zⱼ is the score of token j. For observed token y, the cross-entropy loss is −ln pᵧ. ln uses base e.

**Predict before you run.** Which event would incur greater loss under this distribution, seeing desk or seeing shelf?

In [3]:
import torch
weights = torch.tensor([6., 3., 1.])
logits = weights.log()
p = logits.softmax(0)
target = 1  # door
print([round(float(v), 3) for v in p])
print(round(float(-p[target].log()), 3))

[0.6, 0.3, 0.1]
1.204


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

Shelf. The model assigned it 0.1, giving a loss of about 2.303. Desk at 0.6 would cost about 0.511.

</details>

## Score a whole continuation

Autoregressive means that the model supplies each next-token probability conditional on the tokens before it. Multiply those probabilities to get the continuation’s likelihood. Logs turn a product into a sum. At evaluation time score each observed prefix, including when an earlier greedily generated token would have differed.

Given the same starting word, suppose the model assigns 0.5 to the first observed continuation and 0.25 to the second. The likelihood is 0.125. The mean loss is −ln(0.125)/2. Exponentiating that mean gives perplexity about 2.83.

**Formal notation.** log p(continuation | start) = Σᵢ ln pᵢ. Mean loss L divides the negative sum by the number of predicted tokens. Perplexity is exp(L).

**Predict before you run.** What perplexity results if a model assigns 1/4 to every observed token?

In [4]:
import math
observed = [0.5, 0.25]
log_likelihood = sum(math.log(p) for p in observed)
loss = -log_likelihood / len(observed)
print("likelihood", math.prod(observed))
print("log likelihood, mean loss, perplexity",
      [round(v, 3) for v in (log_likelihood, loss, math.exp(loss))])

likelihood 0.125
log likelihood, mean loss, perplexity [-2.079, 1.04, 2.828]


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

The mean loss is −ln(1/4) = ln(4). Exponentiating gives 4, regardless of sequence length.

</details>

## Fit a tiny language model

Return to two possible continuations, desk and door. The training data contain four entries beginning with open and four beginning with close. Learn a table with one row per starting word and one column per continuation. Fetch the rows, score the observed targets, and update the table by gradient descent. A transformer can replace this one-word lookup while retaining the same loss.

Open is followed by desk three times and door once. Close has the opposite counts. Starting from tied logits gives both tokens 0.5. During training the open row should raise its desk probability and the close row should raise its door probability.

**Formal notation.** Z[c,j] is the score for next token j after context c. The training objective is the mean of −ln p(target | context) over all eight pairs.

**Predict before you run.** Why should extra training on these same counts stop improving near P(desk | open) = 0.75?

In [5]:
import torch
import torch.nn.functional as F
contexts = torch.tensor([0,0,0,0, 1,1,1,1])  # open, close
targets = torch.tensor([0,0,0,1, 0,1,1,1])   # desk, door
Z = torch.nn.Parameter(torch.zeros(2, 2))
optimizer = torch.optim.SGD([Z], lr=1.0)
for step in range(25):
    optimizer.zero_grad()
    loss = F.cross_entropy(Z[contexts], targets)
    loss.backward()
    optimizer.step()
print("P(desk | open), P(desk | close)")
print([round(float(p), 3) for p in Z.detach().softmax(1)[:, 0]])

P(desk | open), P(desk | close)
[0.749, 0.251]


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

Raising that probability helps the three desk examples but hurts the door example. For this unconstrained row, the mean loss is minimized at the observed fractions 3/4 and 1/4.

</details>

## Choose how to decode

Decoding chooses an output using the distribution. Greedy decoding always selects a highest-scoring token, using a tie rule if needed. Sampling draws from the probabilities. A positive temperature divides the logits before softmax. Smaller temperatures concentrate probability on larger logits. Top-k sampling keeps the k highest scores and renormalizes their probabilities.

Begin with probabilities [0.6, 0.3, 0.1]. At temperature 1, greedy picks desk. Sampling with k = 2 gives probabilities [2/3, 1/3, 0]. Temperature 2 first takes square roots of the original weights, making the distribution more even.

**Formal notation.** pⱼ(T) ∝ exp(zⱼ/T). The proportionality sign means divide by the sum afterwards. In top-k sampling only the retained entries appear in that sum.

**Predict before you run.** If k = 1 and the highest score is unique, can a positive temperature change the chosen token?

In [6]:
import torch
z = torch.tensor([6., 3., 1.]).log()
for temp in [1., 2.]:
    probs = (z / temp).softmax(0)
    print(temp, [round(float(p), 3) for p in probs])
top = z.topk(2).indices
masked = torch.full_like(z, -torch.inf)
masked[top] = z[top]
print("top two", [round(float(p), 3) for p in masked.softmax(0)])

1.0 [0.6, 0.3, 0.1]
2.0 [0.473, 0.334, 0.193]
top two [0.667, 0.333, 0.0]


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

No. Dividing all logits by the same positive number preserves their ordering. Only the highest-scoring token survives.

</details>

## Generate and score different things

Generation adds a sampled token to the input and uses this new prefix for the following step. Its choices affect later predictions. Keeping a fixed random seed lets you rerun a particular simulation. The chance of a complete generated path still depends on every transition along that path.

Use a two-word bigram model with rows [0.75, 0.25] and [0.4, 0.6] for desk and door. Starting from desk, the path door then desk has probability 0.25 × 0.4 = 0.1. The code also samples a four-step continuation.

**Formal notation.** tᵢ₊₁ ∼ p(· | tᵢ) means draw the next token from the row for the current token. A real generator may stop at an end token or a length limit.

**Predict before you run.** During held-out evaluation the model prefers desk but the next recorded token is door. What goes into the next prefix?

In [7]:
import torch
rows = torch.tensor([[0.75, 0.25], [0.4, 0.6]])
rng = torch.Generator().manual_seed(12)
words = ["desk", "door"]
path = [0]
for _ in range(4):
    path.append(int(torch.multinomial(rows[path[-1]], 1, generator=rng)))
print([words[i] for i in path])
print("P(door then desk | desk)", round(float(rows[0,1] * rows[1,0]), 3))

['desk', 'desk', 'door', 'door', 'door']
P(door then desk | desk) 0.1


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

Use the recorded door for likelihood evaluation. In free generation use whichever token the decoder chose. These procedures answer different questions.

</details>

## Distinguish pretraining, adaptation and evaluation

Pretraining fits a model on a broad collection of examples. Fine-tuning continues weight updates on selected data for a new use. Prompting supplies examples or instructions as input with the weights fixed. A large language model may use a transformer with many more parameters and a much larger corpus than our table. The same evaluation question remains: which data and capabilities does the score represent?

Suppose fine-tuning raises a desired token’s probability from 0.4 to 0.7 on a held-out desk example, but lowers a library token’s probability from 0.8 to 0.5 on a separate example. Report both losses. Keep evaluation records separate from training and compare perplexities using the same tokens and scoring convention.

**Formal notation.** ΔL = L_after − L_before. A negative change means the observed token received a higher probability. Next-token loss measures predictive fit on these examples. Factual correctness needs a separate check against evidence.

**Predict before you run.** A model has lower perplexity on desk notes than another model, using the same test tokens. Can we conclude it answers desk policy questions correctly?

In [8]:
import math
for context, before, after in [("desk", .4, .7), ("library", .8, .5)]:
    delta = -math.log(after) + math.log(before)
    print(context, "change in observed-token loss", round(delta, 3))

desk change in observed-token loss -0.56
library change in observed-token loss 0.47


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

The result supports better average next-token probability on those notes. Test policy answers directly against verified records before making a claim about accuracy.

</details>

## Exercise 1

With the first token given, list the inputs and targets for red door opens <end>. Use a full-prefix model.

<details><summary>Hint</summary>

Each target follows the last word in its input.

</details>

Check your work.

<details><summary>Reveal answer</summary>

red → door; red door → opens; red door opens → <end>. The model makes three predictions.

</details>

## Exercise 2

Two observed continuation tokens receive probabilities 0.8 and 0.2. Compute likelihood, mean loss and perplexity.

<details><summary>Hint</summary>

Multiply first, then take the negative logarithm and divide by two.

</details>

Check your work.

<details><summary>Reveal answer</summary>

The product is 0.16. The mean loss is −ln(0.16)/2 ≈ 0.9163. Perplexity is exp(0.9163) = 2.5.

</details>

## Exercise 3

For probabilities [0.5, 0.3, 0.2], compare greedy, unrestricted sampling and top-2 sampling at temperature 1.

<details><summary>Hint</summary>

Which outputs are retained? What is their retained mass?

</details>

Check your work.

<details><summary>Reveal answer</summary>

Greedy always picks the first token. Full sampling uses the supplied probabilities. Top-2 sampling retains mass 0.8 and uses [0.625, 0.375, 0].

</details>

## Exercise 4

Two candidates have probabilities 0.8 and 0.2. Apply temperature 2. Which remains the greedy choice?

<details><summary>Hint</summary>

Take square roots and renormalize, or transform their 4:1 odds.

</details>

Check your work.

<details><summary>Reveal answer</summary>

The odds become 2:1, so probabilities become 2/3 and 1/3. The first token remains the greedy choice.

</details>

## Exercise 5

After a weight update, predicting the observed word becomes easier on training records. Design a test for whether the update helps on new requests.

<details><summary>Hint</summary>

Specify what to hold aside, what score to use, and another capability to check.

</details>

Check your work.

<details><summary>Reveal answer</summary>

Hold out whole new requests before fitting, and remove duplicates from training. Score both versions on the same target tokens and compute mean loss. If the goal is correct answers, additionally judge responses against verified records.

</details>

## Exercise 6

A system gets two example question-answer pairs in its prompt. A second system takes gradient steps on those pairs. Which system fine-tuned?

<details><summary>Hint</summary>

Locate the information that persists beyond the current input.

</details>

Check your work.

<details><summary>Reveal answer</summary>

The second system changed its weights. The first received contextual examples with fixed weights. For a fair comparison test on separate questions.

</details>

Generating words meant making one conditional draw at a time. Suppose the desired output is a whole pattern of brightness values. Module 13 compares three ways of learning to produce such observations.

Our generator can choose between desk and door. To generate a two-pixel image instead, what must change?

<details><summary>Reveal answer</summary>

We must define a distribution or sampling process over pairs of pixel values. A latent variable or a sequence of denoising steps can describe dependencies between them.

</details>